In [ ]:
import os
import csv
import json
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, Literal, List

import numpy as np
import torch
from torch import nn
from tqdm import tqdm

import datasets
import unet
import prof_unet
from SplitNet import SplitNet


# -----------------------------------------------------------------------------
# Setup
# -----------------------------------------------------------------------------

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


ModelType = Literal["splitnet_attn", "splitnet", "unet", "attn_unet", "prof_unet"]
EvalDatasetMode = Literal["fixed", "border", "border_pressure"]


# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------

class ChannelSelectDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, channels="KP"):
        self.base_dataset = base_dataset
        self.channels = channels

    def _idx(self):
        if self.channels == "all":
            return [0, 1, 2]
        if self.channels == "KP":
            return [0, 1]
        if self.channels == "K":
            return [0]
        if self.channels == "P":
            return [1]
        if self.channels == "phi":
            return [2]
        raise ValueError(f"Bad channels: {self.channels}")

    def __len__(self):
        return len(self.base_dataset)

    def __getattr__(self, name):
        # lets dataset.types, dataset.sims, dataset.num_steps(), etc still work
        return getattr(self.base_dataset, name)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        chans = self._idx()

        if len(item) == 3:
            feat, label, mask = item
            return feat[chans], label[chans], mask

        feat, label = item
        return feat[chans], label[chans]



@dataclass
class EvalConfig:
    name: str
    model_path: str
    model_type: ModelType

    # This is what the model was trained with.
    # Used only for naming/metadata and mask analysis.
    train_dataset_mode: str = "fixed"
    training_mode: str = "physics_limited"
    darcy_weight: Optional[float] = None

    # This is the dataset used for testing.
    # Default is fixed because your datasets.py says FixedDenseDatasetFull is for testing.
    eval_dataset_mode: EvalDatasetMode = "fixed"

    channels: str = "KP"
    test_sims_path: str = "../test_sims.npy"
    sim_max_exclusive: Optional[int] = 250

    steps: tuple = (0, 200)
    types: tuple = (0,)
    points_per_side: int = 3
    radius: int = 5

    results_dir: str = "test_eval_outputs"
    dataset_kwargs: Optional[Dict[str, Any]] = None


# -----------------------------------------------------------------------------
# Model factory
# -----------------------------------------------------------------------------

def make_model(model_type: ModelType, channels: str = "KP") -> nn.Module:
    if channels == "all":
        num_channels = 3
    elif channels == "KP":
        num_channels = 2
    elif channels in ["K", "P", "phi"]:
        num_channels = 1
    else:
        raise ValueError("channels must be 'all', 'KP', 'K', 'P', or 'phi'.")

    model_type = model_type.lower()

    if model_type == "splitnet_attn":
        if channels != "KP":
            raise ValueError("SplitNet only supports channels='KP'.")
        return SplitNet(attn=True).to(DEVICE)

    if model_type == "splitnet":
        if channels != "KP":
            raise ValueError("SplitNet only supports channels='KP'.")
        return SplitNet(attn=False).to(DEVICE)

    if model_type == "unet":
        return unet.SmallUnet(channels=num_channels).to(DEVICE)

    if model_type == "attn_unet":
        return unet.AttnUnet(channels=num_channels).to(DEVICE)

    if model_type == "prof_unet":
        return prof_unet.UNet(in_channels=num_channels, num_classes=num_channels).to(DEVICE)

    raise ValueError(f"Unknown model_type: {model_type}")


def load_model(config: EvalConfig) -> nn.Module:
    loaded = torch.load(config.model_path, map_location=DEVICE, weights_only=False)

    if isinstance(loaded, dict):
        model = make_model(config.model_type, config.channels)
        model.load_state_dict(loaded)
    else:
        model = loaded.to(DEVICE)

    model.eval()
    return model


# -----------------------------------------------------------------------------
# Dataset factory
# -----------------------------------------------------------------------------

def load_sim_ids(path: str, sim_max_exclusive: Optional[int]) -> np.ndarray:
    sims = np.load(path)
    if sim_max_exclusive is not None:
        sims = sims[sims < sim_max_exclusive]
    return sims


def make_eval_dataset(config: EvalConfig):
    sims = load_sim_ids(config.test_sims_path, config.sim_max_exclusive)

    kwargs = dict(config.dataset_kwargs or {})
    kwargs.setdefault("points_per_side", config.points_per_side)
    kwargs.setdefault("radius", config.radius)
    kwargs.setdefault("steps", config.steps)
    kwargs.setdefault("types", list(config.types))

    # Important:
    # Build the full testing dataset first.
    # Then slice channels afterward with ChannelSelectDataset.
    kwargs["channels"] = "all"

    if config.eval_dataset_mode == "fixed":
        base_dataset = datasets.FixedDenseDatasetFull(sims, **kwargs)

    elif config.eval_dataset_mode == "border":
        base_dataset = datasets.BorderDenseDatasetFull(sims, **kwargs)

    elif config.eval_dataset_mode == "border_pressure":
        base_dataset = datasets.BorderDensePressureGradientDatasetFull(sims, **kwargs)

    else:
        raise ValueError(f"Bad eval_dataset_mode: {config.eval_dataset_mode}")

    return ChannelSelectDataset(base_dataset, channels=config.channels)


# -----------------------------------------------------------------------------
# Physics / metrics
# -----------------------------------------------------------------------------

def darcy_residual_map(out: torch.Tensor) -> torch.Tensor:
    if out.shape[1] < 2:
        raise ValueError("Darcy loss needs at least K and P channels.")

    k = out[:, 0:1]
    p = out[:, 1:2]

    p_y, p_x = torch.gradient(p, dim=(-2, -1))

    flux_y = k * p_y
    flux_x = k * p_x

    div_y = torch.gradient(flux_y, spacing=(1,), dim=(-2,))[0]
    div_x = torch.gradient(flux_x, spacing=(1,), dim=(-1,))[0]

    return div_y + div_x


def darcy_loss_from_output(out: torch.Tensor) -> torch.Tensor:
    return (darcy_residual_map(out) ** 2).mean()


def align_channels(label: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    if label.shape[1] == out.shape[1]:
        return label
    if label.shape[1] > out.shape[1]:
        return label[:, :out.shape[1]]
    raise ValueError(f"Label has {label.shape[1]} channels but output has {out.shape[1]}.")


def modified_error(pred: torch.Tensor, target: torch.Tensor):
    if pred.shape != target.shape:
        raise ValueError(f"Shape mismatch: {pred.shape} vs {target.shape}")

    n = pred.size(0)
    err = (pred - target).pow(2).reshape(n, -1)
    mse = err.mean(dim=1)

    energy = target.pow(2).reshape(n, -1).mean(dim=1)
    nmse = mse / (energy + 1e-8)

    mae = (pred - target).abs().reshape(n, -1).mean(dim=1)
    bias = (pred - target).reshape(n, -1).mean(dim=1)

    return {
        "mse": mse.mean(),
        "nmse": nmse.mean(),
        "energy": energy.mean(),
        "mae": mae.mean(),
        "bias": bias.mean(),
    }


def modified_error_masked(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor):
    if pred.shape != target.shape:
        raise ValueError(f"Shape mismatch: {pred.shape} vs {target.shape}")

    if mask.dim() == 2:
        mask = mask.unsqueeze(0).unsqueeze(0)
    elif mask.dim() == 3:
        mask = mask.unsqueeze(1)

    mask = mask.to(pred.device).float()
    mask = mask.expand(-1, pred.shape[1], -1, -1)

    n = pred.size(0)

    diff = pred - target
    diff2 = diff.pow(2) * mask
    absdiff = diff.abs() * mask
    biasdiff = diff * mask

    denom = mask.reshape(n, -1).sum(dim=1) + 1e-8

    mse = diff2.reshape(n, -1).sum(dim=1) / denom
    energy = (target.pow(2) * mask).reshape(n, -1).sum(dim=1) / denom
    nmse = mse / (energy + 1e-8)
    mae = absdiff.reshape(n, -1).sum(dim=1) / denom
    bias = biasdiff.reshape(n, -1).sum(dim=1) / denom

    return {
        "mse": mse.mean(),
        "nmse": nmse.mean(),
        "energy": energy.mean(),
        "mae": mae.mean(),
        "bias": bias.mean(),
    }


# -----------------------------------------------------------------------------
# Masks
# -----------------------------------------------------------------------------

def build_border_mask(H=200, W=200):
    mask = torch.zeros((H, W), dtype=torch.bool)
    mask[0:5, :] = True
    mask[-5:, :] = True
    return mask


def build_fixed_mask(points_per_side=3, radius=5, H=200, W=200):
    mask = torch.zeros((H, W), dtype=torch.bool)

    div = points_per_side + 1
    point_x = np.arange(W // div, W, W // div)
    point_y = np.arange(H // div, H, H // div)

    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")

    for y0 in point_y:
        for x0 in point_x:
            disk = (yy - int(y0)) ** 2 + (xx - int(x0)) ** 2 <= radius ** 2
            mask |= disk

    return mask


def build_eval_mask(config: EvalConfig, feat: torch.Tensor):
    H, W = feat.shape[-2], feat.shape[-1]

    # This is the observed-region mask you want to compare against.
    # Usually, use the model's training dataset mode.
    mode = config.train_dataset_mode

    if mode == "border" or mode == "border_pressure":
        return build_border_mask(H, W)

    if mode == "fixed":
        return build_fixed_mask(
            points_per_side=config.points_per_side,
            radius=config.radius,
            H=H,
            W=W,
        )

    # fallback: if unknown, use fixed mask
    return build_fixed_mask(
        points_per_side=config.points_per_side,
        radius=config.radius,
        H=H,
        W=W,
    )


# -----------------------------------------------------------------------------
# Main evaluation
# -----------------------------------------------------------------------------

def evaluate_model_to_npz(config: EvalConfig):
    os.makedirs(config.results_dir, exist_ok=True)

    out_npz = os.path.join(config.results_dir, f"eval_{config.name}.npz")
    out_summary_json = os.path.join(config.results_dir, f"eval_{config.name}_summary.json")

    print(f"\n=== Evaluating {config.name} ===")
    print(f"Model path: {config.model_path}")
    print(f"Eval dataset: {config.eval_dataset_mode}")
    print(f"Train mask mode: {config.train_dataset_mode}")

    model = load_model(config)
    dataset = make_eval_dataset(config)

    K = len(dataset.types)
    n_sims = dataset.sims.shape[0]
    T = dataset.num_steps()

    shape = (K, n_sims, T)

    arrays = {
        "mse_total": torch.zeros(shape),
        "nmse_total": torch.zeros(shape),
        "mae_total": torch.zeros(shape),
        "bias_total": torch.zeros(shape),
        "energy_total": torch.zeros(shape),

        "darcy": torch.zeros(shape),

        "mse_inmask": torch.zeros(shape),
        "nmse_inmask": torch.zeros(shape),
        "mae_inmask": torch.zeros(shape),
        "bias_inmask": torch.zeros(shape),

        "mse_outmask": torch.zeros(shape),
        "nmse_outmask": torch.zeros(shape),
        "mae_outmask": torch.zeros(shape),
        "bias_outmask": torch.zeros(shape),

        "mse_k": torch.zeros(shape),
        "nmse_k": torch.zeros(shape),
        "mae_k": torch.zeros(shape),
        "bias_k": torch.zeros(shape),

        "mse_p": torch.zeros(shape),
        "nmse_p": torch.zeros(shape),
        "mae_p": torch.zeros(shape),
        "bias_p": torch.zeros(shape),
    }

    with torch.inference_mode():
        for type_i in tqdm(range(K), desc="types"):
            for sim_i in tqdm(range(n_sims), leave=False, desc="sims"):
                for step_i in range(T):
                    idx = type_i * n_sims * T + sim_i * T + step_i

                    feat, label = dataset[idx]
                    feat = feat.to(DEVICE).unsqueeze(0)
                    label = label.to(DEVICE).unsqueeze(0)

                    out = model(feat)
                    label = align_channels(label, out)

                    obs_mask = build_eval_mask(config, feat[0]).to(DEVICE)
                    out_mask = ~obs_mask

                    total = modified_error(out, label)
                    inmask = modified_error_masked(out, label, obs_mask)
                    outmask = modified_error_masked(out, label, out_mask)

                    k_metrics = modified_error(out[:, 0:1], label[:, 0:1])
                    p_metrics = modified_error(out[:, 1:2], label[:, 1:2])

                    arrays["mse_total"][type_i, sim_i, step_i] = total["mse"].cpu()
                    arrays["nmse_total"][type_i, sim_i, step_i] = total["nmse"].cpu()
                    arrays["mae_total"][type_i, sim_i, step_i] = total["mae"].cpu()
                    arrays["bias_total"][type_i, sim_i, step_i] = total["bias"].cpu()
                    arrays["energy_total"][type_i, sim_i, step_i] = total["energy"].cpu()

                    arrays["darcy"][type_i, sim_i, step_i] = darcy_loss_from_output(out).cpu()

                    arrays["mse_inmask"][type_i, sim_i, step_i] = inmask["mse"].cpu()
                    arrays["nmse_inmask"][type_i, sim_i, step_i] = inmask["nmse"].cpu()
                    arrays["mae_inmask"][type_i, sim_i, step_i] = inmask["mae"].cpu()
                    arrays["bias_inmask"][type_i, sim_i, step_i] = inmask["bias"].cpu()

                    arrays["mse_outmask"][type_i, sim_i, step_i] = outmask["mse"].cpu()
                    arrays["nmse_outmask"][type_i, sim_i, step_i] = outmask["nmse"].cpu()
                    arrays["mae_outmask"][type_i, sim_i, step_i] = outmask["mae"].cpu()
                    arrays["bias_outmask"][type_i, sim_i, step_i] = outmask["bias"].cpu()

                    arrays["mse_k"][type_i, sim_i, step_i] = k_metrics["mse"].cpu()
                    arrays["nmse_k"][type_i, sim_i, step_i] = k_metrics["nmse"].cpu()
                    arrays["mae_k"][type_i, sim_i, step_i] = k_metrics["mae"].cpu()
                    arrays["bias_k"][type_i, sim_i, step_i] = k_metrics["bias"].cpu()

                    arrays["mse_p"][type_i, sim_i, step_i] = p_metrics["mse"].cpu()
                    arrays["nmse_p"][type_i, sim_i, step_i] = p_metrics["nmse"].cpu()
                    arrays["mae_p"][type_i, sim_i, step_i] = p_metrics["mae"].cpu()
                    arrays["bias_p"][type_i, sim_i, step_i] = p_metrics["bias"].cpu()

    np_arrays = {k: v.numpy() for k, v in arrays.items()}
    np.savez(out_npz, **np_arrays)

    summary = summarize_np_arrays(np_arrays)
    summary["config"] = asdict(config)
    summary["npz_path"] = out_npz

    with open(out_summary_json, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved NPZ: {out_npz}")
    print(f"Saved summary: {out_summary_json}")

    del model
    torch.cuda.empty_cache()

    return out_npz, out_summary_json


def summarize_np_arrays(np_arrays: Dict[str, np.ndarray]) -> Dict[str, float]:
    summary = {}

    for key, arr in np_arrays.items():
        summary[f"mean_{key}"] = float(np.nanmean(arr))
        summary[f"median_{key}"] = float(np.nanmedian(arr))
        summary[f"std_{key}"] = float(np.nanstd(arr))

    return summary


# -----------------------------------------------------------------------------
# Batch eval + combined CSV
# -----------------------------------------------------------------------------

def evaluate_many(configs: List[EvalConfig], combined_csv="test_eval_outputs/model_test_summary.csv"):
    summaries = []

    for config in configs:
        _, summary_path = evaluate_model_to_npz(config)

        with open(summary_path, "r") as f:
            summary = json.load(f)

        row = {
            "name": config.name,
            "model_type": config.model_type,
            "train_dataset_mode": config.train_dataset_mode,
            "training_mode": config.training_mode,
            "darcy_weight": config.darcy_weight,
            "eval_dataset_mode": config.eval_dataset_mode,
            "model_path": config.model_path,
            "npz_path": summary["npz_path"],

            "mean_mse_total": summary["mean_mse_total"],
            "mean_nmse_total": summary["mean_nmse_total"],
            "mean_darcy": summary["mean_darcy"],

            "mean_mse_inmask": summary["mean_mse_inmask"],
            "mean_mse_outmask": summary["mean_mse_outmask"],

            "mean_mse_k": summary["mean_mse_k"],
            "mean_mse_p": summary["mean_mse_p"],

            "mean_mae_total": summary["mean_mae_total"],
            "mean_bias_total": summary["mean_bias_total"],
        }

        summaries.append(row)

    os.makedirs(os.path.dirname(combined_csv), exist_ok=True)

    with open(combined_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=summaries[0].keys())
        writer.writeheader()
        writer.writerows(summaries)

    print(f"\nSaved combined CSV: {combined_csv}")


# -----------------------------------------------------------------------------
# Example selected models
# -----------------------------------------------------------------------------

SELECTED_MODELS = [
    # EvalConfig(
    #     name="fixed_splitnet_darcy_0p001",
    #     model_type="splitnet",
    #     train_dataset_mode="fixed",
    #     training_mode="physics_limited",
    #     darcy_weight=0.001,
    #     model_path="recent_analysis_split_na/physics_tests/fixed/fixed_physics_limited_splitnet_darcy_0p001_best_state.pt",
    # ),
    # EvalConfig(
    #     name="fixed_splitnet_darcy_1p0",
    #     model_type="splitnet",
    #     train_dataset_mode="fixed",
    #     training_mode="physics_limited",
    #     darcy_weight=1.0,
    #     model_path="recent_analysis_split_na/physics_tests/fixed/fixed_physics_limited_splitnet_darcy_1p0_best_state.pt",
    # ),
    # EvalConfig(
    #     name="fixed_splitnet_attn_darcy_0p001",
    #     model_type="splitnet_attn",
    #     train_dataset_mode="fixed",
    #     training_mode="physics_limited",
    #     darcy_weight=0.001,
    #     model_path="recent_analysis/physics_tests/fixed/fixed_physics_limited_splitnet_attn_darcy_0p001_best_state.pt",
    # ),
    # EvalConfig(
    #     name="fixed_attn_unet_darcy_0p0",
    #     model_type="attn_unet",
    #     train_dataset_mode="fixed",
    #     training_mode="physics_limited",
    #     darcy_weight=0.0,
    #     channels="KP",
    #     model_path="recent_analysis_unet/physics_tests/fixed/fixed_physics_limited_attn_unet_darcy_0p0_best_state.pt",
    # ),
    # EvalConfig(
    #     name="border_attn_unet_darcy_0p1",
    #     model_type="attn_unet",
    #     train_dataset_mode="border",
    #     training_mode="physics_limited",
    #     darcy_weight=0.1,
    #     channels="KP",
    #     model_path="recent_analysis_unet/physics_tests/border/border_physics_limited_attn_unet_darcy_0p1_best_state.pt",
    # ),
    # EvalConfig(
    #     name="border_pressure_attn_unet_darcy_0p01",
    #     model_type="attn_unet",
    #     train_dataset_mode="border_pressure",
    #     training_mode="physics_limited",
    #     darcy_weight=0.01,
    #     channels="KP",
    #     model_path="recent_analysis_unet/physics_tests/border_pressure/border_pressure_physics_limited_attn_unet_darcy_0p01_best_state.pt",
    # ),
    EvalConfig(
    name="fixed_splitnet_attn_baseline_full_nodarcy",
    model_type="splitnet_attn",
    train_dataset_mode="fixed",
    training_mode="baseline_full",
    darcy_weight=0.0,
    channels="KP",
    model_path="baseline_full/final/fixed_splitnet_attn_baseline_full_nodarcy_best_state.pt",
    ),
    EvalConfig(
        name="fixed_splitnet_baseline_full_nodarcy",
        model_type="splitnet",
        train_dataset_mode="fixed",
        training_mode="baseline_full",
        darcy_weight=0.0,
        channels="KP",
        model_path="baseline_full/final/fixed_splitnet_baseline_full_nodarcy_best_state.pt",
    ),
    EvalConfig(
        name="fixed_attn_unet_baseline_full_nodarcy",
        model_type="attn_unet",
        train_dataset_mode="fixed",
        training_mode="baseline_full",
        darcy_weight=0.0,
        channels="KP",
        model_path="baseline_full/final/fixed_attn_unet_baseline_full_nodarcy_best_state.pt",
    ),
    
]


if __name__ == "__main__":
    evaluate_many(
        SELECTED_MODELS,
        combined_csv="test_eval_outputs/model_test_summary.csv",
    )


=== Evaluating fixed_splitnet_attn_baseline_full_nodarcy ===
Model path: baseline_full/final/fixed_splitnet_attn_baseline_full_nodarcy_best_state.pt
Eval dataset: fixed
Train mask mode: fixed


FileNotFoundError: [Errno 2] No such file or directory: 'baseline_full/final/fixed_splitnet_attn_baseline_full_nodarcy_best_state.pt'